In [0]:
%sql
-- Używamy domyślnego katalogu Unity Catalog w wersji Free (zazwyczaj 'workspace')
USE CATALOG workspace;

CREATE SCHEMA IF NOT EXISTS steam_lakehouse;
USE SCHEMA steam_lakehouse;

-- 1. WARSTWA BRONZE: Surowe dane JSON z API Steama (historia zapytań)
CREATE TABLE IF NOT EXISTS bronze_steam_appdetails (
    app_id INT,
    raw_json STRING,
    ingested_at TIMESTAMP
) USING DELTA;

-- 2. WARSTWA SILVER: Oczyszczony katalog gier i aktualne ceny w PLN
CREATE TABLE IF NOT EXISTS silver_games_catalog (
    app_id INT,
    name STRING,
    is_free BOOLEAN,
    currency STRING,
    initial_price_pln DOUBLE,
    final_price_pln DOUBLE,
    discount_percent INT,
    header_image STRING,
    updated_at TIMESTAMP
) USING DELTA;

-- 3. WARSTWA GOLD: Widok biznesowy pod aplikację budżetową
CREATE OR REPLACE VIEW gold_games_budget_view AS
SELECT 
    app_id,
    name,
    final_price_pln AS current_price,
    initial_price_pln AS regular_price,
    ROUND(initial_price_pln - final_price_pln, 2) AS savings_pln,
    discount_percent,
    CASE 
        WHEN is_free = true THEN 'Darmowa'
        WHEN discount_percent >= 50 THEN 'Super okazja (-50% lub więcej)'
        WHEN discount_percent > 0 THEN 'W promocji'
        ELSE 'Cena regularna'
    END AS deal_status,
    header_image,
    updated_at
FROM silver_games_catalog;